<a href="https://colab.research.google.com/github/fwitschel/ResponsibleAI/blob/main/notebooks/XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Execute this code only if in colab
if 'COLAB_GPU' in os.environ:
  print("Executing in Colab!")
  # Cloning GitHub repository
  !git clone https://github.com/fwitschel/ResponsibleAI.git
  %cd ResponsibleAI

In [ ]:
!pip install lime
import shap
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lime
import lime.lime_tabular
from xgboost import XGBClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn import tree

In [ ]:
# load the data
data = pd.read_csv('/content/ResponsibleAI/data/H1.csv')

# remove the columns corresponding to other complications that we do not want to predict
data = data.drop('DepositType', axis=1)
data = data.drop('ReservationStatus', axis=1)
data = data.drop('ReservationStatusDate', axis=1)
data = data.drop('AssignedRoomType', axis=1)
data = data.drop('BookingChanges', axis=1)
data = data.drop('DaysInWaitingList', axis=1)
data = data.drop('ArrivalDateYear', axis=1)
#data = data.drop('Country', axis=1)
data = data.drop('Agent', axis=1)
data = data.drop('Company', axis=1)

X = data.drop('IsCanceled', axis=1)
#one_hot_X = pd.get_dummies(X)
y = data.IsCanceled

# mark categorical variables as categorical
cat_cols = X.select_dtypes(include='object').columns

# Missing categorical values become their own category
X[cat_cols] = X[cat_cols].fillna('Missing').astype('category')

# Missing numerical values can still become 0
num_cols = X.columns.difference(cat_cols)
X[num_cols] = X[num_cols].fillna(0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = XGBClassifier(enable_categorical=True)
model.fit(X_train, y_train)

In [ ]:
j = 0
k = 0
thresh = 0.8

# find an example where the prediction is 1:
preds = model.predict_proba(X_test)
for i in range(42):
    if (preds[i])[0]>thresh:
        j = i
    if (preds[i])[1]>thresh:
        k = i
print("prediction for instance", j, "in test data:", model.predict_proba(X_test)[j])
print("prediction for instance", k, "in test data:", model.predict_proba(X_test)[k])

In [ ]:
# Fits the explainer
explainer = shap.TreeExplainer(model)
# Calculates the SHAP values - may take some time
shap_values = explainer.shap_values(X_test)

In [ ]:
# Calculate SHAP values for the test set, producing Explanation objects
shap_values_explanation = explainer(X_test)

# Make SHAP waterfall plots for instances j and k
print(f"SHAP waterfall plot for instance j (index: {j}):")
shap.plots.waterfall(shap_values_explanation[j])

print(f"\nSHAP waterfall plot for instance k (index: {k}):")
shap.plots.waterfall(shap_values_explanation[k])

In [ ]:
# --- Prepare data for LIME ---

# Make numerical copies for LIME
X_train_lime = X_train.copy()
X_test_lime = X_test.copy()

categorical_features = []
categorical_names = {}

for col in cat_cols:
    col_idx = X_train.columns.get_loc(col)
    categorical_features.append(col_idx)

    # Get the categories already defined in the pandas categorical column
    categories = list(X_train[col].cat.categories)
    categorical_names[col_idx] = categories

    # Replace categories by 0, 1, 2, ... for LIME
    X_train_lime[col] = X_train[col].cat.codes
    X_test_lime[col] = X_test[col].cat.codes


# LIME will call this function with its numerical representation.
# Convert category codes back into pandas categorical variables
# before asking XGBoost for predictions.
def lime_predict(x):
    x = pd.DataFrame(x, columns=X_train.columns)

    for col in cat_cols:
        col_idx = X_train.columns.get_loc(col)
        categories = categorical_names[col_idx]

        x[col] = pd.Categorical.from_codes(
            x[col].astype(int),
            categories=categories
        )

    return model.predict_proba(x)


# --- Create LIME explainer ---

explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train_lime.values,
    feature_names=X_train.columns.tolist(),
    class_names=['not cancelled', 'cancelled'],
    categorical_features=categorical_features,
    categorical_names=categorical_names,
    verbose=True,
    mode='classification',
    random_state=42
)


# --- Explain first chosen instance ---

exp = explainer.explain_instance(
    X_test_lime.iloc[j].values,
    lime_predict,
    num_features=X_train.shape[1]
)

print("LIME score:", exp.score)
print("LIME local prediction:", exp.local_pred)
print("XGBoost prediction:", preds[j])

exp.show_in_notebook(show_table=True)


# --- Explain second chosen instance ---

exp = explainer.explain_instance(
    X_test_lime.iloc[k].values,
    lime_predict,
    num_features=X_train.shape[1]
)

print("LIME score:", exp.score)
print("LIME local prediction:", exp.local_pred)
print("XGBoost prediction:", preds[k])

exp.show_in_notebook(show_table=True)